# 02CollectPostingDetail

- 목적: Replay and parse all 29 verified local Linkareer SSR pages
- 담당 Agent: `P4-A1-SOURCE`
- Stage ID: `A1-02-DETAIL`
- 입력: `crawl/observed_inputs/OBSERVED_INPUT_20260806_01/HANDOFF.json`
- 처리: `audit_input_manifest;replay_observed_raw;extract_detail_record;write_stage_artifacts` 모듈 호출만 수행
- 출력: 29건 posting_detail_replay CSV/Parquet와 replay QA 및 4개 종료 artifact
- 선행 Gate: `INDEX_FIXTURE_APQ_READY`
- 후속 활용: A1-03-ASSET metadata candidate routing 및 Agent 2 observed parser

이 Notebook은 orchestration-only이며 empirical analysis와 production promotion을 활성화하지 않는다.

In [ ]:
RUN_MODE = "observed-dev"
AGENT_ID = "P4-A1-SOURCE"
STAGE_ID = "A1-02-DETAIL"
CONTRACT_VERSION = "2.1.2"
SCHEMA_VERSION = "posting-manifest-v1"
DATA_VERSION = "observed-dev-20260806.1"
CRAWL_RELEASE_ID = "CRAWL_20260806_03"
AS_OF_DATE = "2026-08-06"
INPUT_MANIFEST_PATH = "crawl/observed_inputs/OBSERVED_INPUT_20260806_01/HANDOFF.json"
OUTPUT_ROOT = "crawl/runs/notebooks/observed-dev/AGENT1_20260806_01"
RANDOM_SEED = 20260806
FAIL_ON_GATE = True
EMPIRICAL_ANALYSIS_ALLOWED = False

## Imports and isolated observed-development environment

In [ ]:
import os
from pathlib import Path
import sys

def locate_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "crawl" / "src" / "p4_crawl").is_dir():
            return candidate
        nested = candidate / "DSJA" / "project_4"
        if (nested / "crawl" / "src" / "p4_crawl").is_dir():
            return nested
    raise RuntimeError("Could not locate DSJA/project_4")

PROJECT_ROOT = locate_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "crawl" / "src"))
from p4_crawl.config import RunConfig
from p4_crawl.observed import require_repository_relative

assert RUN_MODE == "observed-dev"
assert CONTRACT_VERSION == "2.1.2"
assert CRAWL_RELEASE_ID == "CRAWL_20260806_03"
assert EMPIRICAL_ANALYSIS_ALLOWED is False
assert Path(OUTPUT_ROOT).as_posix().startswith("crawl/runs/notebooks/observed-dev/")
require_repository_relative(INPUT_MANIFEST_PATH)
require_repository_relative(OUTPUT_ROOT)

CRAWL_ROOT = PROJECT_ROOT / "crawl"
RAW_SOURCE_ROOT = Path(os.environ.get("P4_CRAWL_RAW_SOURCE_ROOT", str(CRAWL_ROOT))).resolve()
if not (RAW_SOURCE_ROOT / "data/raw").is_dir():
    RAW_SOURCE_ROOT = CRAWL_ROOT
RUN_ROOT = PROJECT_ROOT / OUTPUT_ROOT
STAGE_ROOT = RUN_ROOT / STAGE_ID
INPUT_MANIFEST = PROJECT_ROOT / INPUT_MANIFEST_PATH
run_id = Path(OUTPUT_ROOT).relative_to("crawl/runs").as_posix()
config = RunConfig(
    project_root=PROJECT_ROOT, run_id=run_id, run_mode=RUN_MODE,
    contract_version=CONTRACT_VERSION, crawl_release_id=CRAWL_RELEASE_ID,
    data_version=DATA_VERSION, as_of_date=AS_OF_DATE, random_seed=RANDOM_SEED,
)
PARAMETERS = {name: globals()[name] for name in [
    "RUN_MODE", "AGENT_ID", "STAGE_ID", "CONTRACT_VERSION", "SCHEMA_VERSION",
    "DATA_VERSION", "CRAWL_RELEASE_ID", "AS_OF_DATE", "INPUT_MANIFEST_PATH",
    "OUTPUT_ROOT", "RANDOM_SEED", "FAIL_ON_GATE", "EMPIRICAL_ANALYSIS_ALLOWED",
]}

def quality_row(gate, rule, severity, status, observed, threshold, evidence):
    return {
        "gateId": gate, "ruleId": rule, "severity": severity, "status": status,
        "observedValue": observed, "threshold": threshold,
        "evidencePath": f"{OUTPUT_ROOT}/{STAGE_ID}/{evidence}",
    }

## Input and checksum audit

In [ ]:
from p4_crawl.observed import audit_input_manifest

input_audit = audit_input_manifest(PROJECT_ROOT, INPUT_MANIFEST_PATH, CRAWL_RELEASE_ID)
assert input_audit["contractVersion"] == CONTRACT_VERSION
input_audit

## Stage module call

In [ ]:
STAGE_WARNING = 'Only the 29 observed raw pages are replayed; this is not full-corpus detail coverage.'
from p4_crawl.observed import replay_observed_raw

OBSERVED_ROOT = INPUT_MANIFEST.parent
metrics = replay_observed_raw(PROJECT_ROOT, OBSERVED_ROOT, STAGE_ROOT, raw_source_root=RAW_SOURCE_ROOT)
quality = [
    quality_row("CRAWL_OBSERVED_INPUT_READY", "RAW_REPLAY", "ERROR", "PASS" if metrics["rawReplayPassed"] == 29 and metrics["rawReplayFailed"] == 0 else "FAIL", metrics["rawReplayPassed"], 29, "raw_replay_metrics.json"),
    quality_row("ACTIVITY_TEXT_RECOVERED", "SSR_APOLLO_PARSE", "ERROR", "PASS" if metrics["activityTextRecovered"] == 29 else "FAIL", metrics["activityTextRecovered"], 29, "posting_detail_replay.parquet"),
    quality_row("RAW_PII_NOT_PERSISTED", "PII_POLICY", "ERROR", "PASS" if not metrics["managerPiiPersisted"] else "FAIL", metrics["managerPiiPersisted"], False, "raw_replay_metrics.json"),
    quality_row("ACTIVITY_TEXT_AMBIGUOUS_AUTO_SELECTION", "FALLBACK_POLICY", "ERROR", "PASS" if metrics["activityTextAmbiguousAutoSelected"] == 0 else "FAIL", metrics["activityTextAmbiguousAutoSelected"], 0, "raw_replay_metrics.json"),
]
persisted_files = [STAGE_ROOT / name for name in ["posting_detail_replay.parquet", "posting_detail_replay.csv", "raw_replay_failures.json", "raw_replay_metrics.json"]]

## Termination artifacts and gate result

In [ ]:
from p4_crawl.stage import write_stage_artifacts

release_blocked = STAGE_ID == "A1-04-RELEASE" and not bool(metrics.get("crawlReleaseReady"))
manifest = write_stage_artifacts(
    config=config, stage_id=STAGE_ID, schema_version=SCHEMA_VERSION,
    started_at=f"{AS_OF_DATE}T00:00:00+09:00", parameters=PARAMETERS,
    input_manifest_path=INPUT_MANIFEST, stage_root=STAGE_ROOT,
    metric_values=metrics, quality_rows=quality, persisted_files=persisted_files,
    warnings=[STAGE_WARNING],
    status_override="NOT_EVALUATED" if release_blocked else None,
)
if FAIL_ON_GATE and (any(row["status"] == "FAIL" for row in quality) or release_blocked):
    raise RuntimeError(f"{STAGE_ID} quality gate failed")
{"stageId": STAGE_ID, "status": manifest["status"], "metrics": metrics, "artifacts": manifest["terminationArtifacts"]}

Observed-development interpretation: Only the 29 observed raw pages are replayed; this is not full-corpus detail coverage. Analysis and production readiness remain disabled.